# Report tinkering — CFFL_A 2025

Run all cells to load the archive and inspect every silver table. Set `CURRENT_WEEK` to an integer to inspect an earlier week; `None` uses the metadata week. Re-run the setup cell after changing the week or source files because tables are cached.


In [3]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Works when launched from the repository root or a folder inside it.
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "report_code" / "processor.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from report_code import ReportProcessor

CURRENT_WEEK = 10  # e.g. 5; None uses current_week/last_collected_week in metadata.
DATA_PATH = REPO_ROOT / "yahoo-fantasy-data" / "data" / "CFFL_A" / "2025"
report = ReportProcessor(DATA_PATH, current_week=CURRENT_WEEK)
print(f"{report.metadata['league_name']} | {report.metadata['season']} | Week {report.current_week}")
report.teams


Conarroe Fantasy A League | 2025 | Week 10


,team_id,team_name,manager
0,1,Kline is Mine,Kline is Mine
1,10,Team Bows,Team Bows
2,11,The Show,The Show
3,12,Coach,Coach
4,2,Barry Mash,Barry Mash
5,3,Bill Cosby's Sleepers,Bill Cosby's Sleepers
6,4,Should Be In C League,Should Be In C League
7,5,Five Star Man,Five Star Man
8,6,Gruesome Twosome,Gruesome Twosome
9,7,Lonesome Onesome,Lonesome Onesome


## `silver_divisions`

Latest division assignments through the selected week.


In [7]:
silver_divisions = report.silver_divisions
print(f"{len(silver_divisions):,} rows × {len(silver_divisions.columns):,} columns")
display(silver_divisions)


12 rows × 5 columns


,team_id,team_name,division_id,division_name,week
0,1,Kline is Mine,1,Elite Divison,17
1,12,Coach,1,Elite Divison,17
2,6,Gruesome Twosome,1,Elite Divison,17
3,8,Master of Coin,1,Elite Divison,17
4,4,Should Be In C League,2,Middle America,17
5,5,Five Star Man,2,Middle America,17
6,7,Lonesome Onesome,2,Middle America,17
7,9,RVIII,2,Middle America,17
8,10,Team Bows,3,Basically B League,17
9,11,The Show,3,Basically B League,17


## `silver_player`

Core player fields plus weekly actual-point ranks within each team / FA pool for every league position, including flex. Ties share rank; reserves and ineligible players have blank ranks. `is_optimal` marks one best legal lineup per pool, or is blank when optimization is unavailable.


In [8]:
silver_player = report.silver_player
silver_player

,player_id,player_name,week,position,eligible_positions,nfl_team,bye_week,team_id,team_name,roster_slot,is_starting,actual_points,projected_points,draft_round,draft_pick,draft_team_id,is_stud,is_dud,volatility_known,rank_def,rank_k,rank_qb,rank_rb,rank_te,rank_w_r_t,rank_wr,is_optimal
0,100001,Falcons,1,DEF,[DEF],Atl,5,<NA>,NaN,NaN,False,1.00,4.94,NaN,NaN,<NA>,False,False,True,17,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False
1,100002,Bills,1,DEF,[DEF],Buf,7,<NA>,NaN,NaN,False,0.00,5.49,14.0,167.0,12,False,False,True,19,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False
2,100003,Bears,1,DEF,[DEF],Chi,5,<NA>,NaN,NaN,False,11.00,6.51,NaN,NaN,<NA>,False,False,True,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False
3,100004,Bengals,1,DEF,[DEF],Cin,10,12,Coach,DEF,True,7.00,7.43,NaN,NaN,<NA>,False,False,True,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True
4,100005,Browns,1,DEF,[DEF],Cle,9,<NA>,NaN,NaN,False,4.00,4.93,NaN,NaN,<NA>,False,False,True,11,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21245,8937,Josh Johnson,9,QB,[QB],Cin,10,<NA>,NaN,NaN,False,0.00,0.00,NaN,NaN,<NA>,False,False,True,<NA>,<NA>,13,<NA>,<NA>,<NA>,<NA>,False
21246,9265,Matthew Stafford,9,QB,[QB],LAR,8,3,Bill Cosby's Sleepers,QB,True,26.84,16.55,15.0,175.0,3,False,False,True,<NA>,<NA>,1,<NA>,<NA>,<NA>,<NA>,True
21247,9526,Graham Gano,9,K,[K],NYG,14,<NA>,NaN,NaN,False,5.00,6.83,NaN,NaN,<NA>,False,False,True,<NA>,11,<NA>,<NA>,<NA>,<NA>,<NA>,False
21248,9547,Brian Hoyer,9,QB,[QB],LV,8,<NA>,NaN,NaN,False,0.00,0.00,NaN,NaN,<NA>,False,False,True,<NA>,<NA>,13,<NA>,<NA>,<NA>,<NA>,False


## `silver_schedule`

Team opponents by week. Includes future weeks from the selected schedule snapshot.


In [9]:
silver_schedule = report.silver_schedule
print(f"{len(silver_schedule):,} rows × {len(silver_schedule.columns):,} columns")
print(f"{len(silver_player):,} rows × {len(silver_player.columns):,} columns")

display(silver_schedule)


168 rows × 3 columns
21,250 rows × 27 columns


,team_id,week,opponent_id
0,1,1,12
1,10,1,11
2,11,1,10
3,12,1,1
4,2,1,3
...,...,...,...
163,5,14,9
164,6,14,8
165,7,14,4
166,8,14,6


## `silver_reconciliation`

Starter totals compared with official Yahoo scores loaded automatically from `report.bronze_points_recon` (`points recon/`). Joined by team and week; `difference` is calculated minus official points.


In [2]:
silver_reconciliation = report.silver_reconciliation
silver_reconciliation

,team_id,week,calculated_points,starter_count,missing_points,roster_starter_count,official_points,difference,status
0,1,1,96.68,9,0,9,96.68,-1.421085e-14,matched
1,1,2,103.08,9,0,9,103.08,0.000000e+00,matched
2,1,3,95.12,9,0,9,95.12,-1.421085e-14,matched
3,1,4,65.72,9,0,9,65.72,0.000000e+00,matched
4,1,5,132.96,9,0,9,132.96,0.000000e+00,matched
...,...,...,...,...,...,...,...,...,...
199,9,13,83.14,9,0,9,83.14,0.000000e+00,matched
200,9,14,78.06,9,0,9,78.06,0.000000e+00,matched
201,9,15,94.90,9,0,9,94.90,0.000000e+00,matched
202,9,16,80.10,9,0,9,80.10,0.000000e+00,matched


## `silver_season_reconciliation`

Season totals and reconciliation status through the selected week.


In [6]:
silver_season_reconciliation = report.silver_season_reconciliation
print(f"{len(silver_season_reconciliation):,} rows × {len(silver_season_reconciliation.columns):,} columns")
display(silver_season_reconciliation)


12 rows × 7 columns


,team_id,calculated_points,official_points,matched_weeks,issue_weeks,difference,status
0,1,1717.70,1717.70,17,0,0.00,matched
1,10,1724.74,1724.74,17,0,0.00,matched
2,11,1446.76,1401.68,16,1,45.08,incomplete_or_mismatch
3,12,1694.76,1601.98,16,1,92.78,incomplete_or_mismatch
4,2,2267.34,2267.34,17,0,0.00,matched
5,3,1741.64,1741.64,17,0,0.00,matched
6,4,1705.86,1602.68,16,1,103.18,incomplete_or_mismatch
7,5,1678.44,1599.70,16,1,78.74,incomplete_or_mismatch
8,6,1693.10,1693.10,17,0,0.00,matched
9,7,1636.26,1636.26,17,0,0.00,matched


## `silver_team_week`

Team scores, opponent scores, and results by week.


In [7]:
silver_team_week = report.silver_team_week
print(f"{len(silver_team_week):,} rows × {len(silver_team_week.columns):,} columns")
display(silver_team_week)


204 rows × 12 columns


,team_id,week,actual_points,starter_count,missing_points,roster_starter_count,official_points,difference,status,opponent_id,opponent_points,win
0,1,1,96.68,9,0,9,96.68,-1.421085e-14,matched,12,108.36,0.0
1,1,2,103.08,9,0,9,103.08,0.000000e+00,matched,8,120.54,0.0
2,1,3,95.12,9,0,9,95.12,-1.421085e-14,matched,6,79.02,1.0
3,1,4,65.72,9,0,9,65.72,0.000000e+00,matched,3,104.26,0.0
4,1,5,132.96,9,0,9,132.96,0.000000e+00,matched,4,155.58,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
199,9,13,83.14,9,0,9,83.14,0.000000e+00,matched,7,111.02,0.0
200,9,14,78.06,9,0,9,78.06,0.000000e+00,matched,5,85.80,0.0
201,9,15,94.90,9,0,9,94.90,0.000000e+00,matched,NaN,NaN,NaN
202,9,16,80.10,9,0,9,80.10,0.000000e+00,matched,NaN,NaN,NaN


## `silver_lineups`

Optimal lineups and projection-based lineup selections by team and week.


In [ ]:
silver_lineups = report.silver_lineups
print(f"{len(silver_lineups):,} rows × {len(silver_lineups.columns):,} columns")
display(silver_lineups)


204 rows × 7 columns


,team_id,week,optimal_points,optimal_player_indices,projected_player_indices,yahoo_actual_points,using_yahoo
0,1,1,110.32,"[11, 473, 346, 696, 196, 107, 667, 801, 802]","[11, 473, 797, 196, 696, 667, 61, 436, 802]",88.88,False
1,1,2,127.60,"[11266, 11723, 12047, 11531, 11446, 11917, 119...","[11266, 11723, 11596, 11446, 11946, 11917, 120...",103.08,True
2,1,3,128.02,"[12500, 12973, 13297, 12781, 12696, 12607, 129...","[12500, 12973, 12846, 13196, 12696, 13167, 133...",71.40,False
3,1,4,96.42,"[13750, 14223, 14547, 13946, 14446, 13857, 144...","[13750, 14223, 14547, 13946, 14446, 14417, 140...",85.72,False
4,1,5,132.96,"[15021, 16243, 15346, 15407, 15281, 15107, 151...","[15021, 16243, 15346, 15196, 15407, 15107, 156...",123.54,False
...,...,...,...,...,...,...,...
199,9,13,99.14,"[5006, 6018, 5103, 5282, 5524, 5214, 5266, 543...","[5006, 6018, 5103, 5266, 5524, 5214, 5435, 579...",90.04,False
200,9,14,80.76,"[6256, 7268, 6353, 6516, 6420, 6464, 6685, 704...","[6256, 7268, 6353, 6516, 6774, 6464, 6420, 668...",78.06,True
201,9,15,124.20,"[7505, 8518, 8091, 8343, 7766, 8336, 7670, 793...","[7505, 8518, 8091, 7766, 8024, 7714, 7670, 776...",94.90,True
202,9,16,99.60,"[8755, 9768, 9341, 8920, 9593, 8964, 9526, 918...","[8755, 9768, 8853, 9016, 9274, 9586, 9185, 894...",81.36,False


## Explore a team and week

Edit these values to narrow the player table. All full tables remain available in the variables above.


In [9]:
TEAM_ID = "1"  # Use "FA" for unrostered players.
WEEK = report.current_week

pool = silver_player.team_id.isna() if TEAM_ID == "FA" else silver_player.team_id.eq(TEAM_ID)
rank_columns = [column for column in silver_player if column.startswith("rank_")]
players = silver_player.loc[pool & silver_player.week.eq(WEEK)]
display(players[[
    "player_name", "position", "roster_slot", "actual_points",
    *rank_columns, "is_starting", "is_optimal",
]].sort_values("actual_points", ascending=False))


,player_name,position,roster_slot,actual_points,rank_def,rank_k,rank_qb,rank_rb,rank_te,rank_w_r_t,rank_wr,is_starting,is_optimal
10696,Chase Brown,RB,RB,27.60,<NA>,<NA>,<NA>,1,<NA>,1,<NA>,True,True
10346,Justin Herbert,QB,QB,15.14,<NA>,<NA>,1,<NA>,<NA>,<NA>,<NA>,True,True
10107,Hunter Henry,TE,TE,12.40,<NA>,<NA>,<NA>,<NA>,1,2,<NA>,True,True
10078,Jason Myers,K,K,10.00,<NA>,1,<NA>,<NA>,<NA>,<NA>,<NA>,True,True
10407,Rico Dowdle,RB,BN,7.80,<NA>,<NA>,<NA>,2,<NA>,3,<NA>,False,True
10007,Lions,DEF,DEF,7.00,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,True
10196,Saquon Barkley,RB,RB,6.80,<NA>,<NA>,<NA>,3,<NA>,4,<NA>,True,True
10460,Michael Carter,RB,W/R/T,5.30,<NA>,<NA>,<NA>,4,<NA>,5,<NA>,True,False
10281,David Montgomery,RB,BN,5.00,<NA>,<NA>,<NA>,5,<NA>,6,<NA>,False,False
10297,Darius Slayton,WR,BN,4.60,<NA>,<NA>,<NA>,<NA>,<NA>,7,1,False,True


## Source snapshots

Files used by the tables above.


In [10]:
display(pd.DataFrame([
    {"source": source, "file": filename}
    for source, filenames in report.source_files.items()
    for filename in filenames
]))


,source,file
0,team_data,/home/joe/Workspaces/FF/Fantasy_football_codin...
1,team_data,/home/joe/Workspaces/FF/Fantasy_football_codin...
2,team_data,/home/joe/Workspaces/FF/Fantasy_football_codin...
3,team_data,/home/joe/Workspaces/FF/Fantasy_football_codin...
4,team_data,/home/joe/Workspaces/FF/Fantasy_football_codin...
...,...,...
67,points_recon,/home/joe/Workspaces/FF/Fantasy_football_codin...
68,points_recon,/home/joe/Workspaces/FF/Fantasy_football_codin...
69,points_recon,/home/joe/Workspaces/FF/Fantasy_football_codin...
70,points_recon,/home/joe/Workspaces/FF/Fantasy_football_codin...


## Generate the HTML report

Run the setup cell first. This exports all report sections for the selected `DATA_PATH` and `CURRENT_WEEK` to `report_code/output`. Open the printed path in your browser. Re-running replaces the report for the same league, season, and week.


In [ ]:
odds = report.gold_playoff_odds
if odds.attrs.get("unavailable_reason"):
    print(f"Playoff odds unavailable: {odds.attrs['unavailable_reason']}")

OUTPUT_DIR = REPO_ROOT / "docs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

report_name = f"{DATA_PATH.parent.name.lower()}_{DATA_PATH.name}_week{report.current_week}"
html_path = report.write_html(OUTPUT_DIR / f"{report_name}.html")
print(f"HTML report saved to: {html_path.resolve()}")


HTML report saved to: /home/joe/Workspaces/FF/Fantasy_football_coding/report_code/output/cffl_a_2025_week10.html
